# Basic 03 - Plot and Export
Small, step-by-step plotting/export flow with explicit file-load metadata.


## 1) Imports


In [43]:
from pathlib import Path
import sys
import warnings
import logging

import pandas as pd
from IPython.display import display


## 2) Quiet logs (optional)


In [44]:
warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)


## 3) Make local package importable


In [45]:
def _ensure_local_package() -> None:
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]
    for root in search_roots:
        if (root / "isa_phm").is_dir() and (root / "pyproject.toml").exists():
            root_s = str(root)
            if root_s not in sys.path:
                sys.path.insert(0, root_s)
            return
    raise RuntimeError("Could not locate python-wrapper root with isa_phm package.")

_ensure_local_package()


## 4) Import wrapper


In [46]:
from isa_phm import ISAWrapper


## 5) Pick ISA JSON


In [47]:
ISA_JSON = Path(r"G:\ISA\Datasets\Milling\Data_mill\Multi Run Milling ISA-PHM.json")
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


ISA-JSON : G:\ISA\Datasets\Milling\Data_mill\Multi Run Milling ISA-PHM.json
DATA_ROOT: G:\ISA\Datasets\Milling\Data_mill
Exists   : True


## 6) Build wrapper (performance knobs shown explicitly)


In [48]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    enable_chunked_large_file_mode=True,
    large_file_threshold_mb=64.0,
    chunk_rows=250_000,
)


## 7) Pick first study


In [49]:
study = wrapper.study(wrapper.list_studies()[0].title)
study.title


'Case 1'

## 8) Pick first assay


In [50]:
assay = study.assay(study.list_assays()[0].assay_id)
assay.assay_id


'se01'

## 9) Pick first run


In [51]:
run_id = assay.list_runs()[0].run_id
run_id


'run_01'

## 10) Load one run with file metadata


In [52]:
df, meta = assay.load_dataframe_with_meta(run_id=run_id, file_type="auto")


DataFileError: Cannot decode 'G:\ISA\Datasets\Milling\Data_mill\Case_01\Sensors\vib_table\Case_01_vib_table_run_01.csv'. Tried UTF-8 and Latin-1.

## 11) Show resolved load metadata


In [ ]:
display(pd.DataFrame([meta.model_dump()]))


## 12) Quick data preview


In [ ]:
display(df.head(10))


## 13) Timeseries plot


In [ ]:
fig_ts = assay.plot_timeseries(run_id=run_id, file_type="auto")
fig_ts


## 14) Frequency-domain plot


In [ ]:
fig_fft = assay.plot_frequency_domain(run_id=run_id, file_type="auto")
fig_fft


## 15) Lifecycle features for this assay


In [ ]:
lc = assay.lifecycle_features(file_type="auto")
display(lc.head(10))


## 16) Multi-sensor merged dataframe


In [ ]:
multi_df = study.load_multi_sensor_dataframe(file_type="auto")
display(multi_df.head(10))


## 17) Labeled export dataframe


In [ ]:
labeled_df = study.export_labeled_dataset(file_type="auto")


## 18) Labeled export preview


In [ ]:
display(labeled_df.head(10))
display(pd.DataFrame([{"rows": int(labeled_df.shape[0]), "cols": int(labeled_df.shape[1])}]))


## 19) Validation summary before downstream export


In [ ]:
validation = wrapper.validate_dataset(check_files=True)
display(pd.DataFrame([{
    "ok": validation.ok,
    "n_errors": validation.n_errors,
    "n_warnings": validation.n_warnings,
    "n_info": validation.n_info,
}]))


## 20) AI context export (metadata-only)


In [ ]:
ai_ctx = wrapper.ai_context(include_semantics=True, include_validation=True)
list(ai_ctx.keys())
